In [ ]:
import pandas as pd
from tqdm import tqdm
import numpy as np
import json
import os
from openai import OpenAI, AsyncOpenAI, OpenAIError
import asyncio
from tqdm.asyncio import tqdm_asyncio
import pandas as pd
import openai
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

from miti_behavioral_counts import MITI_BEHAVIOR_COUNTS_PROMPT 
PROJECT_DIR = "/path/to/project"
api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
#Validation Transcripts
df = pd.read_csv("/path/to/project/data/raw/main_socialmedia/chats_raw.csv")
df["id"] = df["session_id"] + "_" + df["order"].astype(str)

In [ ]:
phrase = "Hi! In this interview I want to learn more about how you spend your time"

bad_session_ids = df.loc[
    df["content"].str.contains(phrase, na=False),
    "session_id"
].unique()

df_no_control = df[~df["session_id"].isin(bad_session_ids)].reset_index(drop=True)


In [ ]:
df_interviewer = df_no_control[df_no_control["type"] == "question"]
print(len(df_interviewer))

In [ ]:
global_prompt = MITI_BEHAVIOR_COUNTS_PROMPT

In [ ]:
# 1. Define your parameters and prompt template
# Note: 'messages' is replaced by 'input' based on your requirements for the /v1/responses endpoint
base_params = {
    "model": "gpt-5-nano-2025-08-07",
    "text": {"verbosity": "low"},  # low, medium, high
    "reasoning": {"effort": "low"},   # minimal, low, medium, high
    "store": False,
    "max_output_tokens": 500,
}

# Example global prompt placeholder - replace with your actual prompt string
global_prompt = MITI_BEHAVIOR_COUNTS_PROMPT

def create_batch_file(df, global_prompt, output_filename="batch_requests.jsonl"):
    """
    Creates a JSONL file for OpenAI Batch API using the specified dataframe and parameters.
    """
    
    with open(output_filename, 'w') as f:
        for _, row in df.iterrows():
            # Format the input text using the global prompt
            input_text = global_prompt.format(text=row['content'])
            
            # Create a fresh copy of params for this row and add the input
            body_payload = base_params.copy()
            body_payload["input"] = input_text
            
            # Construct the final request object
            request_object = {
                "custom_id": str(row['id']),
                "method": "POST",
                "url": "/v1/responses",
                "body": body_payload
            }
            
            # Write line to .jsonl file
            f.write(json.dumps(request_object) + '\n')
            
    print(f"Successfully created {output_filename} with {len(df)} requests.")

def create_batch_files(df, global_prompt, output_prefix, num_parts=4):
    """
    Splits the dataframe into evenly sized parts and writes one JSONL per part.
    """
    chunks = np.array_split(df, num_parts)
    for part_index, chunk in enumerate(chunks, start=1):
        output_filename = f"{output_prefix}_part{part_index}.jsonl"
        create_batch_file(chunk, global_prompt, output_filename)



In [ ]:
# Usage
create_batch_files(df_interviewer, MITI_BEHAVIOR_COUNTS_PROMPT, "output/batch_input_v001", num_parts=4)